# V4 FINAL — Precificação Dinâmica com Redes Neurais

## Objetivo
Esta versão consolida as correções apontadas na avaliação da V3 e deixa os resultados **reprodutíveis e auditáveis**.

### Correções incorporadas
- Correção do cálculo de demanda: removida a divisão indevida por 7.
- Validação temporal com `TimeSeriesSplit(n_splits=3)` e seeds 42, 7 e 123.
- Modelo MLP armazenado explicitamente por produto para evitar reutilização acidental no gráfico de elasticidade.
- Comparação com baseline linear no mesmo conjunto de teste.
- Simulação de preço em uma grade de -20% a +20% do preço do concorrente, preservando a lógica de escala da demanda validada na V3.
- ROI e payback calculados no próprio notebook, sem números financeiros hard-coded.
- Documentação da evolução V2 → V3 → V4 e das limitações.

> **Importante:** os resultados financeiros são uma simulação de receita no conjunto de teste. A premissa de margem incremental de 25% é uma hipótese de negócio para a PoC, não uma margem contábil observada na base.


In [ ]:

# ============================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# O arquivo pode estar no Colab ou em /mnt/data.
candidatos = [
    "/content/Aryadne Fernandes_dados_precificacao_dinamica_oficial.csv",
    "/mnt/data/Aryadne Fernandes_dados_precificacao_dinamica_oficial.csv",
]
arquivo = next((p for p in candidatos if os.path.exists(p)), None)

if arquivo is None:
    raise FileNotFoundError(
        "Não encontrei a base oficial. Faça upload de "
        "'Aryadne Fernandes_dados_precificacao_dinamica_oficial.csv'."
    )

df = pd.read_csv(arquivo)
df["data"] = pd.to_datetime(df["data"])
df = df.sort_values(["produto", "data"]).reset_index(drop=True)

print(f"Base carregada: {arquivo}")
print(f"Linhas: {len(df):,} | Colunas: {len(df.columns)}")
print("Produtos:", ", ".join(df["produto"].unique()))

# ============================================================
# 2. ENGENHARIA DE FEATURES
# ============================================================
df["razao_preco"] = df["preco_praticado"] / df["preco_concorrente"]
df["gap_preco_relativo"] = (
    (df["preco_praticado"] - df["preco_concorrente"])
    / df["preco_concorrente"]
)
df["cobertura_estoque_dias"] = (
    df["estoque"] / (df["historico_vendas_7d"] + 1e-5)
)
df["tendencia_vendas"] = (
    df["historico_vendas_7d"] / (df["historico_vendas_30d"] + 1e-5)
)

# Codificação cíclica de mês e dia da semana.
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12.0)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12.0)
df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_da_semana"] / 7.0)
df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_da_semana"] / 7.0)
df["eh_fim_semana"] = df["dia_da_semana"].isin([5, 6]).astype(int)

features_num = [
    "preco_praticado", "preco_concorrente", "razao_preco",
    "gap_preco_relativo", "cobertura_estoque_dias",
    "tendencia_vendas", "historico_vendas_7d",
    "historico_vendas_30d", "estoque", "mes_sin", "mes_cos",
    "dia_semana_sin", "dia_semana_cos", "eh_fim_semana"
]
features_cat = ["periodo_mes"]
features = features_num + features_cat
target = "demanda"

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
])

# ============================================================
# 3. VALIDAÇÃO ROBUSTA — TIME SERIES + MULTI-SEED
# ============================================================
seeds = [42, 7, 123]
produtos = df["produto"].unique()
resultados_validacao = []

print("\n" + "=" * 80)
print("VALIDAÇÃO ROBUSTA")
print("=" * 80)

for prod in produtos:
    df_p = df[df["produto"] == prod].reset_index(drop=True)
    X = df_p[features]
    y = df_p[target].values

    r2_vals, mape_vals = [], []

    for seed in seeds:
        tscv = TimeSeriesSplit(n_splits=3)

        for train_idx, test_idx in tscv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            modelo = Pipeline([
                ("prep", preprocessor),
                ("mlp", MLPRegressor(
                    hidden_layer_sizes=(64, 32),
                    activation="relu",
                    solver="adam",
                    max_iter=1000,
                    early_stopping=True,
                    random_state=seed
                ))
            ])

            modelo.fit(X_train, y_train)
            pred = np.clip(modelo.predict(X_test), 0, None)

            r2_vals.append(r2_score(y_test, pred))
            mape_vals.append(
                mean_absolute_percentage_error(y_test + 1e-5, pred) * 100
            )

    resultados_validacao.append({
        "Produto": prod,
        "R² Médio": np.mean(r2_vals),
        "R² Desvio": np.std(r2_vals),
        "MAPE Médio (%)": np.mean(mape_vals),
        "MAPE Desvio (%)": np.std(mape_vals),
    })

validacao_df = pd.DataFrame(resultados_validacao)
display(validacao_df)

# ============================================================
# 4. SIMULAÇÃO FINAL — 80/20 TEMPORAL
# ============================================================
tabela_negocio = []
modelos = {}

for prod in produtos:
    df_p = df[df["produto"] == prod].reset_index(drop=True)
    split_idx = int(len(df_p) * 0.8)

    df_train = df_p.iloc[:split_idx].copy()
    df_test = df_p.iloc[split_idx:].copy().reset_index(drop=True)

    # Baseline linear
    modelo_linear = Pipeline([
        ("prep", preprocessor),
        ("reg", LinearRegression())
    ])
    modelo_linear.fit(df_train[features], df_train[target].values)
    pred_linear = np.clip(modelo_linear.predict(df_test[features]), 0, None)

    # Rede neural MLP
    modelo_mlp = Pipeline([
        ("prep", preprocessor),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
    modelo_mlp.fit(df_train[features], df_train[target].values)

    # Armazena explicitamente o modelo de cada produto.
    modelos[prod] = {
        "mlp": modelo_mlp,
        "linear": modelo_linear
    }

    pred_mlp = np.clip(modelo_mlp.predict(df_test[features]), 0, None)

    # Métricas no mesmo conjunto de teste.
    r2_linear = r2_score(df_test[target], pred_linear)
    r2_mlp = r2_score(df_test[target], pred_mlp)
    mape_linear = mean_absolute_percentage_error(
        df_test[target] + 1e-5, pred_linear
    ) * 100
    mape_mlp = mean_absolute_percentage_error(
        df_test[target] + 1e-5, pred_mlp
    ) * 100

    receita_historica = (
        df_test["preco_praticado"] * df_test["demanda"]
    ).sum()

    receita_heuristica = 0.0
    receita_ia = 0.0

    for _, row in df_test.iterrows():
        preco_conc = row["preco_concorrente"]
        estoque_atual = row["estoque"]

        # Heurística Se-Então da V3:
        # estoque alto -> desconto de 5%
        # estoque baixo -> aumento de 4%
        if row["cobertura_estoque_dias"] > 15:
            preco_h = row["preco_praticado"] * 0.95
        elif row["cobertura_estoque_dias"] < 5:
            preco_h = row["preco_praticado"] * 1.04
        else:
            preco_h = row["preco_praticado"]

        # Modelo de demanda corrigido: sem divisão indevida por 7.
        fator_demanda_real = row["demanda"] / (
            row["historico_vendas_7d"] + 1e-5
        )

        # A heurística é comparada por meio do baseline linear,
        # mantendo a mesma lógica validada na V3.
        cenario_heuristica = pd.DataFrame(
            [row[features].to_dict()]
        )
        cenario_heuristica["preco_praticado"] = preco_h
        cenario_heuristica["razao_preco"] = preco_h / preco_conc
        cenario_heuristica["gap_preco_relativo"] = (
            (preco_h - preco_conc) / preco_conc
        )

        demanda_heuristica = np.clip(
            modelo_linear.predict(cenario_heuristica[features])[0],
            0,
            None
        ) * fator_demanda_real

        receita_heuristica += preco_h * demanda_heuristica

        # Busca do preço ótimo na faixa de 80% a 120% do concorrente.
        precos_candidatos = np.linspace(
            preco_conc * 0.80,
            preco_conc * 1.20,
            41
        )

        cenarios = pd.DataFrame([row[features].to_dict()] * len(precos_candidatos))
        cenarios["preco_praticado"] = precos_candidatos
        cenarios["razao_preco"] = precos_candidatos / preco_conc
        cenarios["gap_preco_relativo"] = (
            (precos_candidatos - preco_conc) / preco_conc
        )

        demanda_prevista = np.clip(
            modelo_mlp.predict(cenarios[features]),
            0,
            None
        )

        receita_cenarios = precos_candidatos * demanda_prevista
        idx_otimo = np.argmax(receita_cenarios)

        # Converte a previsão do modelo para a escala de demanda observada
        # no registro, preservando a correção de escala da V3.
        demanda_otima_real = (
            demanda_prevista[idx_otimo] * fator_demanda_real
        )

        receita_ia += (
            precos_candidatos[idx_otimo] * demanda_otima_real
        )

    tabela_negocio.append({
        "Produto": prod,
        "Receita Histórica": receita_historica,
        "Receita Heurística": receita_heuristica,
        "Receita IA": receita_ia,
        "Ganho Heurístico (%)": (
            (receita_heuristica / receita_historica) - 1
        ) * 100,
        "Ganho Financeiro IA": receita_ia - receita_historica,
        "Ganho IA (%)": (
            (receita_ia / receita_historica) - 1
        ) * 100,
        "R² Linear": r2_linear,
        "R² MLP": r2_mlp,
        "MAPE Linear (%)": mape_linear,
        "MAPE MLP (%)": mape_mlp,
    })

negocio_df = pd.DataFrame(tabela_negocio)

print("\n" + "=" * 80)
print("RESULTADOS DE NEGÓCIO")
print("=" * 80)
display(negocio_df)

# ============================================================
# 5. ROI E PAYBACK — CÁLCULO REPRODUTÍVEL
# ============================================================
CUSTO_SETUP = 45_000.00
CUSTO_RUN_ANUAL = 54_000.00
MARGEM_INCREMENTAL = 0.25

ganho_receita_periodo = negocio_df["Ganho Financeiro IA"].sum()

# Janela efetiva coberta pelo teste temporal.
datas_inicio = []
datas_fim = []

for prod in produtos:
    df_p = df[df["produto"] == prod].reset_index(drop=True)
    split_idx = int(len(df_p) * 0.8)
    datas_inicio.append(df_p.iloc[split_idx]["data"])
    datas_fim.append(df_p.iloc[-1]["data"])

data_min_teste = min(datas_inicio)
data_max_teste = max(datas_fim)
dias_simulados = max((data_max_teste - data_min_teste).days + 1, 1)

# Annualização explícita: transforma o ganho observado na janela de teste
# em uma estimativa equivalente de 365 dias.
ganho_receita_anualizado = ganho_receita_periodo * (365 / dias_simulados)
margem_incremental_anual = ganho_receita_anualizado * MARGEM_INCREMENTAL

custo_total_ano1 = CUSTO_SETUP + CUSTO_RUN_ANUAL
lucro_incremental_ano1 = margem_incremental_anual - custo_total_ano1
roi_ano1 = (lucro_incremental_ano1 / custo_total_ano1) * 100

margem_mensal_equivalente = margem_incremental_anual / 12
payback_meses = CUSTO_SETUP / margem_mensal_equivalente
payback_dias = payback_meses * 30.44

roi_df = pd.DataFrame([{
    "Ganho receita no teste": ganho_receita_periodo,
    "Dias da janela de teste": dias_simulados,
    "Ganho receita anualizado": ganho_receita_anualizado,
    "Margem incremental anual (25%)": margem_incremental_anual,
    "Custo setup": CUSTO_SETUP,
    "Custo run anual": CUSTO_RUN_ANUAL,
    "Custo total Ano 1": custo_total_ano1,
    "Resultado líquido Ano 1": lucro_incremental_ano1,
    "ROI Ano 1 (%)": roi_ano1,
    "Payback (meses)": payback_meses,
    "Payback (dias)": payback_dias,
}])

print("\n" + "=" * 80)
print("ROI E PAYBACK — CÁLCULO REPRODUTÍVEL")
print("=" * 80)
display(roi_df)

print(f"Ganho de receita no período simulado: R$ {ganho_receita_periodo:,.2f}")
print(f"Janela de simulação: {dias_simulados} dias")
print(f"Ganho de receita anualizado: R$ {ganho_receita_anualizado:,.2f}")
print(f"Margem incremental anual (25%): R$ {margem_incremental_anual:,.2f}")
print(f"Custo total Ano 1: R$ {custo_total_ano1:,.2f}")
print(f"Resultado incremental líquido Ano 1: R$ {lucro_incremental_ano1:,.2f}")
print(f"ROI Ano 1: {roi_ano1:,.1f}%")
print(f"Payback estimado: {payback_meses:,.2f} meses ({payback_dias:,.0f} dias)")


## 5. Curvas de elasticidade e preço ótimo

O gráfico abaixo usa **o modelo MLP específico de cada produto** (`modelos[prod]["mlp"]`). Para um dia representativo do conjunto de teste, são simulados preços entre 80% e 120% do preço do concorrente.

A curva mostra a receita esperada (`preço × demanda prevista`) e o ponto de máximo da grade simulada.

**Cuidado de interpretação:** esta é uma simulação de cenário, não uma estimativa causal de elasticidade. O preço ótimo deve ser validado em piloto controlado antes de uso operacional.


In [ ]:

# ============================================================
# 6. GRÁFICOS DE ELASTICIDADE — MODELO ESPECÍFICO POR PRODUTO
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, prod in enumerate(produtos):
    df_p = df[df["produto"] == prod].reset_index(drop=True)
    split_idx = int(len(df_p) * 0.8)
    df_test = df_p.iloc[split_idx:].copy().reset_index(drop=True)

    modelo_mlp = modelos[prod]["mlp"]

    exemplo = df_test.iloc[0]
    preco_conc = exemplo["preco_concorrente"]

    precos = np.linspace(preco_conc * 0.80, preco_conc * 1.20, 100)

    cenarios = pd.DataFrame(
        [exemplo[features].to_dict()] * len(precos)
    )
    cenarios["preco_praticado"] = precos
    cenarios["razao_preco"] = precos / preco_conc
    cenarios["gap_preco_relativo"] = (
        (precos - preco_conc) / preco_conc
    )

    demanda = np.clip(
        modelo_mlp.predict(cenarios[features]),
        0,
        None
    )
    receita = precos * demanda

    idx_max = np.argmax(receita)
    preco_otimo = precos[idx_max]

    ax = axes[idx]
    ax.plot(precos, receita, linewidth=2.5, label="Receita prevista")
    ax.axvline(
        preco_otimo,
        linestyle="--",
        linewidth=1.5,
        label=f"Ótimo: R$ {preco_otimo:.2f}"
    )
    ax.set_title(prod)
    ax.set_xlabel("Preço simulado")
    ax.set_ylabel("Receita esperada")
    ax.grid(alpha=0.25)
    ax.legend()

plt.suptitle(
    "Curvas de receita esperada por preço — MLP específico por produto",
    fontsize=14,
    fontweight="bold"
)
plt.tight_layout()
plt.show()


## 7. Conclusão, evolução e limitações

### O que foi corrigido
1. **Escala da demanda:** retirada a divisão indevida por 7, evitando inflação artificial dos resultados.
2. **Validação:** `TimeSeriesSplit` com três seeds, preservando a ordem temporal.
3. **Reprodutibilidade:** resultados financeiros, ROI e payback passaram a ser calculados no notebook.
4. **Modelos por produto:** cada produto mantém seu próprio MLP para as curvas de elasticidade.
5. **Comparação:** baseline linear calculado no mesmo conjunto de teste.
6. **Rastreabilidade:** a metodologia de anualização do ROI está explícita no código.

### Principais limitações
- A simulação estima **receita**, não lucro real.
- A margem incremental de 25% é uma premissa da PoC.
- A análise não modela resposta competitiva endógena, custos por produto, restrições comerciais, logística ou capacidade.
- A escolha do preço ótimo é baseada em cenários previstos pelo modelo e **não comprova causalidade**.
- Antes de produção, recomenda-se um piloto controlado comparando a recomendação da IA com a política tradicional.

### Recomendação
Iniciar um piloto de 30 dias em uma filial ou carteira B2B, com grupo de controle e acompanhamento de receita, volume, margem e taxa de conversão. Só após essa validação escalar a solução para outras carteiras e integrar ao processo comercial.
